#Data inspection

In [ ]:
from openpyxl import load_workbook

wb = load_workbook('Online Retail.xlsx')
print('sheet names: ',wb.sheetnames)

ws = wb.active
print("active sheet name: ",ws)
print("total rows: ", ws.max_row)
print("total columns: ", ws.max_column)

for row in ws.iter_rows(min_row =1, max_row=5, values_only=True):
  print(row)

sheet names:  ['Online Retail']
active sheet name:  <Worksheet "Online Retail">
total rows:  541910
total columns:  8
('InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country')
(536365, '85123A', 'WHITE HANGING HEART T-LIGHT HOLDER', 6, datetime.datetime(2010, 12, 1, 8, 26), 2.55, 17850, 'United Kingdom')
(536365, 71053, 'WHITE METAL LANTERN', 6, datetime.datetime(2010, 12, 1, 8, 26), 3.39, 17850, 'United Kingdom')
(536365, '84406B', 'CREAM CUPID HEARTS COAT HANGER', 8, datetime.datetime(2010, 12, 1, 8, 26), 2.75, 17850, 'United Kingdom')
(536365, '84029G', 'KNITTED UNION FLAG HOT WATER BOTTLE', 6, datetime.datetime(2010, 12, 1, 8, 26), 3.39, 17850, 'United Kingdom')


In [1]:
import pandas as pd

df = pd.read_excel('Online Retail.xlsx')



In [2]:
expected_columns = ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

missing_columns = []

for column in expected_columns:
  if column not in df.columns:
    missing_columns.append(column)

if missing_columns:
  print('missing columns are: ', missing_columns)
else:
  print("all columns are correct.")

all columns are correct.


#data profiling

In [ ]:
df.info()
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB
            Quantity                    InvoiceDate      UnitPrice  \
count  541909.000000                         541909  541909.000000   
mean        9.552250  2011-07-04 13:34:57.156386048       4.611114   
min    -80995.000000            2010-12-01 08:26:00  -11062.060000   
25%         1.000000        

In [ ]:
print("missing description orders : ", df['Description'].isnull().sum())
print("negative quantity orders   : ", (df['Quantity']<0).sum())
print("zero quantity orders       : ", (df['Quantity']==0).sum())
print("negative price orders      : ", (df['UnitPrice']<0).sum())
print("zero price orders          : ", (df['UnitPrice']==0).sum())
print("cancelled orders           :", (df['InvoiceNo'].astype(str).str.startswith('C')).sum())
print("duplicate rows             :", df.duplicated().sum())

missing description orders :  1454
negative quantity orders   :  10624
zero quantity orders       :  0
negative price orders      :  2
zero price orders          :  2515
cancelled orders           : 9288
duplicate rows             : 5268


In [ ]:
print("missing customers id count :", df['CustomerID'].isnull().sum())

missing customers id count : 135080


#unique column values

In [ ]:
print("unique customers count            : ", df['CustomerID'].nunique())
print("unique orders count               : ", df['InvoiceNo'].nunique())
print("unique products codes count       : ", df['StockCode'].nunique())
print("unique products description count : ", df['Description'].nunique())


unique customers count            :  4372
unique orders count               :  25900
unique products codes count       :  4070
unique products description count :  4223


#missing values and cancelled orders check

In [3]:
#==============================================
# missing description
#==============================================

count1 = df[df['Description'].isnull() & (df['UnitPrice']==0)].shape
print("missing description and zero price rows count: ", count1)

#==============================================
# cancelled orders
#==============================================

count2 = df[df['InvoiceNo'].astype(str).str.startswith('C') & (df['Quantity']<0)].shape
print("cancelled orders and negative quantity: ", count2)

cancelled_df = df['InvoiceNo'].astype(str).str.startswith('C')

#==============================================
# remaining negative quantity apart from cancelled orders
#==============================================

temp_df = df[~cancelled_df]
print("\n-----------------------------------------------\n")
print(temp_df[temp_df['Quantity']<0].head(10))

print(temp_df[temp_df['Quantity']<0].tail(10))


missing description and zero price rows count:  (1454, 8)
cancelled orders and negative quantity:  (9288, 8)

-----------------------------------------------

     InvoiceNo StockCode Description  Quantity         InvoiceDate  UnitPrice  \
2406    536589     21777         NaN       -10 2010-12-01 16:50:00        0.0   
4347    536764    84952C         NaN       -38 2010-12-02 14:42:00        0.0   
7188    536996     22712         NaN       -20 2010-12-03 15:30:00        0.0   
7189    536997     22028         NaN       -20 2010-12-03 15:30:00        0.0   
7190    536998     85067         NaN        -6 2010-12-03 15:30:00        0.0   
7192    537000     21414         NaN       -22 2010-12-03 15:32:00        0.0   
7193    537001     21653         NaN        -6 2010-12-03 15:33:00        0.0   
7195    537003     85126         NaN        -2 2010-12-03 15:33:00        0.0   
7196    537004     21814         NaN       -30 2010-12-03 15:34:00        0.0   
7197    537005     21692       

In [4]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates.sort_values(['InvoiceNo', 'StockCode']).head(10))

    InvoiceNo StockCode                       Description  Quantity  \
494    536409     21866       UNION JACK FLAG LUGGAGE TAG         1   
517    536409     21866       UNION JACK FLAG LUGGAGE TAG         1   
485    536409     22111      SCOTTIE DOG HOT WATER BOTTLE         1   
539    536409     22111      SCOTTIE DOG HOT WATER BOTTLE         1   
489    536409     22866     HAND WARMER SCOTTY DOG DESIGN         1   
527    536409     22866     HAND WARMER SCOTTY DOG DESIGN         1   
521    536409     22900   SET 2 TEA TOWELS I LOVE LONDON          1   
537    536409     22900   SET 2 TEA TOWELS I LOVE LONDON          1   
565    536412     21448         12 DAISY PEGS IN WOOD BOX         2   
578    536412     21448         12 DAISY PEGS IN WOOD BOX         1   

            InvoiceDate  UnitPrice  CustomerID         Country  
494 2010-12-01 11:45:00       1.25     17908.0  United Kingdom  
517 2010-12-01 11:45:00       1.25     17908.0  United Kingdom  
485 2010-12-01 11:45:00

#cleaning

In [5]:
#==============================================
# make a copy
#==============================================

df_clean = df.copy()

#==============================================
# shape before cleaning
#==============================================

print("raw rows             : ", df.shape)
print("cancelled rows count :",cancelled_df.sum())
print("clean working rows   : ", df_clean.shape)

raw rows             :  (541909, 8)
cancelled rows count : 9288
clean working rows   :  (541909, 8)


In [6]:
#==============================================
# remove duplicates
#==============================================

duplicates_removed = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()

print('\n===================== Duplicate Rows=====================\n')
print("Duplicates Removed :", duplicates_removed)
print("\nCurrent Shape :", df_clean.shape)

#==============================================
# remove cancelled
#==============================================
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print('\n=====================cancelled orders=====================\n')
print('cancelled rows count: ', cancelled_df.sum())
print('current shape :', df_clean.shape)

#==============================================
# remove negative quantities
#==============================================
negative_quantity = (df_clean['Quantity']<0).sum()
df_clean = df_clean[~(df_clean['Quantity']<0)]
print('\n=====================negative qunatity=====================\n')
print('negative qunatity rows : ', negative_quantity)
print('current shape          :', df_clean.shape)

#==============================================
# remove negative prices
#==============================================
negative_price = (df_clean['UnitPrice']<0).sum()
df_clean = df_clean[~(df_clean['UnitPrice']<0)]
print('\n=====================negative qunatity=====================\n')
print('negative price rows : ', negative_price)
print('current shape          :', df_clean.shape)

#==============================================
# remove zero prices
#==============================================
zero_price = (df_clean['UnitPrice']==0).sum()
df_clean = df_clean[~(df_clean['UnitPrice']==0)]
print('\n=====================negative qunatity=====================\n')
print('zero price rows : ', negative_price)
print('current shape   :', df_clean.shape)


===================== Duplicate Rows=====================

Duplicates Removed : 5268

Current Shape : (536641, 8)

=====================cancelled orders=====================

cancelled rows count:  9288
current shape : (527390, 8)

=====================negative qunatity=====================

negative qunatity rows :  1336
current shape          : (526054, 8)

=====================negative qunatity=====================

negative price rows :  2
current shape          : (526052, 8)

=====================negative qunatity=====================

zero price rows :  2
current shape   : (524878, 8)


#business metrics

In [7]:
df_clean['Revenue'] = df_clean['UnitPrice']*df_clean['Quantity']

# =====================================================
# Date Features
# =====================================================

df_clean["Year"] = (df_clean["InvoiceDate"].dt.year)
df_clean["Month"] = (df_clean["InvoiceDate"].dt.month)
df_clean["MonthName"] = (df_clean["InvoiceDate"].dt.month_name())


total_Revenue = df_clean['Revenue'].sum()
total_Quantity = df_clean['Quantity'].sum()
total_orders = df_clean['InvoiceNo'].nunique()
total_products = df_clean['StockCode'].nunique()
total_customers = df_clean['CustomerID'].nunique()

#Reports

In [8]:
# =====================================================
# monthly summary
# =====================================================

monthly_summary  = (df_clean.groupby(['Year','Month','MonthName'])
                  .agg(TotalRevenue=("Revenue", "sum"),
        TotalOrders=("InvoiceNo", "nunique"),
        ProductsSold=("Quantity", "sum"),
        UniqueCustomers=("CustomerID", "nunique"))
                  .reset_index())

monthly_summary = (monthly_summary.sort_values(by=["Year", "Month"]))

# =====================================================
# product summary
# =====================================================

product_revenue =( df_clean.groupby('Description')
                  .agg(totalRevenue=('Revenue','sum'),
                       totalQuantity=('Quantity','sum'),
                       totalOrders=('InvoiceNo','nunique'))
                  .sort_values(by='totalRevenue',ascending=False)
)

# =====================================================
# country summary
# =====================================================


country_summary = (df_clean.groupby('Country')
                    .agg(totalRevenue=('Revenue','sum'),
                         totalorders=('InvoiceNo','nunique'),
                         totalCustomers=('CustomerID','nunique'),
                         productsSold=('Quantity','sum')))
top_countries = (country_summary.sort_values(by='totalRevenue',ascending=False))


#adding MonthYear column to monthly summary

In [55]:
monthly_summary["MonthYear"] = (
    monthly_summary["MonthName"].str[:3]
    + " "
    + monthly_summary["Year"].astype(str)
)

monthly_summary = monthly_summary[
    ["Year", "Month", "MonthName", "MonthYear",
     "TotalRevenue", "TotalOrders",
     "ProductsSold", "UniqueCustomers"]
]

#writing to excel file

In [56]:
cancelled_df = df[df['InvoiceNo'].astype(str).str.startswith('C')]


with pd.ExcelWriter('Retail_Automated_Report.xlsx', engine='openpyxl') as writer:

    # Summary
    summary = pd.DataFrame({
        'Metric': [
            'Total Revenue',
            'Total Quantity',
            'Total Orders',
            'Total Products',
            'Total Customers'
        ],
        'Value': [
            total_Revenue,
            total_Quantity,
            total_orders,
            total_products,
            total_customers
        ]
    })

    summary.to_excel(writer, sheet_name='Summary', index=False)

    # Other reports
    monthly_summary.to_excel(
        writer,
        sheet_name='Monthly Report',
        index=False
    )

    product_revenue.to_excel(
        writer,
        sheet_name='Product Report'
    )

    top_countries.to_excel(
        writer,
        sheet_name='Country Report'
    )

    cancelled_df.to_excel(
        writer,
        sheet_name='Cancellations',
        index=False
    )

#loading workbook

In [57]:
from openpyxl import load_workbook

wb = load_workbook('Retail_Automated_Report.xlsx')

print(wb.sheetnames)

['Summary', 'Monthly Report', 'Product Report', 'Country Report', 'Cancellations']


# cell properties

In [58]:
from openpyxl.styles import Font, PatternFill, Alignment

header_font = Font(bold=True, color="FFFFFF")
header_fill = PatternFill(
    start_color="1F4E78",
    end_color="1F4E78",
    fill_type="solid"
)
header_alignment = Alignment(horizontal="center")

In [59]:
for ws in wb.worksheets:
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment

# freeze panes

In [60]:
for ws in wb.worksheets:
    ws.freeze_panes = "A2"

#column widths

In [74]:
for ws in wb.worksheets:

    for column in ws.columns:

        max_length = 0
        column_letter = column[0].column_letter

        for cell in column:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))

        ws.column_dimensions[column_letter].width = max_length + 4

wb.save('Retail_Automated_Report.xlsx')

#number fomatting

In [62]:
# =====================================================
# formatting Summary sheet
# =====================================================

ws = wb['Summary']
for row in range(2, ws.max_row+1):
  metric = ws.cell(row,1).value
  if 'Revenue' in str(metric):
    ws.cell(row,2).number_format = '$#,##0.00'
  else:
    ws.cell(row,2).number_format = '#,##0'

# =====================================================
# formatting rest of the sheet
# =====================================================

for ws_name in ['Monthly Report', 'Product Report', 'Country Report']:
  ws = wb[ws_name]
  for cell in ws[1]:
    header = str(cell.value)

    if 'Revenue' in header:
      for row in range(2, ws.max_row+1):
        ws.cell(row,cell.column).number_format = '$#,##0.00'

    elif 'orders' in header.lower():
      for row in range(2, ws.max_row+1):
       ws.cell(row,cell.column).number_format = '#,##0'

    elif 'products' in header.lower():
      for row in range(2, ws.max_row+1):
       ws.cell(row,cell.column).number_format = '#,##0'

    elif 'quantity' in header.lower():
      for row in range(2, ws.max_row+1):
       ws.cell(row,cell.column).number_format = '#,##0'

#creating tables and table styles

In [63]:
from openpyxl.worksheet.table import Table, TableStyleInfo

for ws_name in ['Monthly Report', 'Product Report', 'Country Report']:

  ws = wb[ws_name]

  table_name = f"{ws_name.replace(" ", "")}Table2"
  tab = ws.tables.get(table_name)

  if tab is not None:
    print(ws_name, ' table already exist')
  else:
    last_cell = ws.cell(ws.max_row, ws.max_column)
    ref = f"A1:{last_cell.coordinate}"

    tab = Table(displayName=table_name, ref = ref)

    ws.add_table(tab)

  style = TableStyleInfo(name = 'TableStyleMedium2',
                         showFirstColumn=False,
                         showLastColumn=False,
                         showRowStripes= True,
                         showColumnStripes=False)

  tab.tableStyleInfo = style

wb.save("Retail_Automated_Report.xlsx")

#creating charts

In [72]:
from openpyxl.chart import LineChart, BarChart, Reference

#==================================================
# function for charts
#==================================================

def create_chart(ws, chart_type, data_col, category_col,
                 min_row_data, min_row_cat, max_row, title, position):

  if chart_type == 'line':
    chart = LineChart()

  elif chart_type == 'bar':
    chart = BarChart()

  data = Reference( ws, min_col = data_col, min_row = min_row_data,
                    max_row = max_row)

  categories = Reference(ws, min_col = category_col, min_row = min_row_cat,
                    max_row = max_row)

  chart.add_data(data, titles_from_data=True)
  chart.set_categories(categories)

  chart.title = title
  chart.title.overlay = False

  chart.width = 17
  chart.height = 7

  chart.series[0].marker.symbol = 'circle'
  chart.series[0].marker.size = 6
  ws.add_chart(chart,position)

#==================================================
# list for automation of charts for more than 1 sheets
#==================================================


charts_config = [{'sheet': 'Monthly Report', 'type':'line',
                  'data_col':5, 'category_col':4, 'min_row_data':1, 'min_row_cat':2,'max_row':14,
                  'title':'Monthly Revenue Tred', 'position':'J2'},
                 {'sheet': 'Product Report', 'type':'bar',
                  'data_col':2, 'category_col':1, 'min_row_data':1, 'min_row_cat':2, 'max_row':11,
                  'title':'Top 10 Products by Revenue', 'position':'F2'},
                 {'sheet': 'Country Report', 'type':'bar',
                  'data_col':2, 'category_col':1,'min_row_data':1, 'min_row_cat':2, 'max_row':11,
                  'title':'Top 10 Countries by Revenue', 'position':'G2'}]

#==================================================
# for loop to call function
#==================================================

for config in charts_config:
  ws = wb[config['sheet']]
  create_chart(ws, config['type'], config['data_col'], config['category_col'],
               config['min_row_data'], config['min_row_cat'], config['max_row'], config['title'], config['position'])


wb.save('Retail_Automated_Report.xlsx')